# Extracting features and embeddings of TUNI Emotion Dataset




## 1. Setup


In [1]:
import types, importlib, sys
imp = types.ModuleType("imp")
imp.reload = importlib.reload
sys.modules["imp"] = imp
%load_ext autoreload
%autoreload 2


In [ ]:
import msclap
import tensorflow_hub
print("OK")


In [ ]:
import os 
import sys
import pandas as pd
from pathlib import Path

# Add parent directory to path so we can import scripts module
parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)


In [ ]:
from tuni_data_config import tuni_emotion_dataset_config
from tuni_extraction_config import tuni_emotion_extraction
from scripts.validate_config import validate_configs

# Create dataset instance by calling the function
tuni_emotion_dataset = tuni_emotion_dataset_config()

print(f"Dataset: {tuni_emotion_dataset.name}")
print(f"Root directory: {tuni_emotion_dataset.root_dir}")
print(f"Levels: {tuni_emotion_dataset.level_names}")

# View extraction configuration
print(f"\nExtractors: {tuni_emotion_extraction.extractors}")
print(f"Output directory: {tuni_emotion_extraction.output_dir}")
print(f"Window lengths: {tuni_emotion_extraction.window_lengths}")

# Validate configurations
try:
    validate_configs(tuni_emotion_dataset, tuni_emotion_extraction)
    print("\n✓ Configurations are valid!")
except ValueError as e:
    print(f"\n✗ Configuration error: {e}")


## 2. Run Extractions

Run **one** extraction cell below (prefer `console_output=True` for progress). Full pipeline may take several hours. See [README.md](README.md) in this folder for Mac setup and troubleshooting.


In [ ]:
import logging
from scripts.run_extraction import run_extraction
from pathlib import Path

# Run extraction (this may take a while depending on dataset size)
output_dir = Path(tuni_emotion_extraction.output_dir)
log_file = str(output_dir / "tuni_extraction_run_log.jsonl")

# Pass file paths to run_extraction (not the config objects)
run_extraction(
    dataset_config_path="tuni_data_config.py",
    extraction_config_path="tuni_extraction_config.py",
    log_file=log_file,
    console_output=False, # aim to reduce memory use of VSCode so the session doesnt crash
)

print(f"✓ Extraction complete! Logs saved to {log_file}")


In [ ]:
import logging
from scripts.run_extraction import run_extraction
from pathlib import Path

# Run extraction (this may take a while depending on dataset size)
output_dir = Path(tuni_emotion_extraction.output_dir)
log_file = str(output_dir / "tuni_extraction_run_log.jsonl")

# Pass file paths to run_extraction (not the config objects)
result = run_extraction(
    dataset_config_path="tuni_data_config.py",
    extraction_config_path="tuni_extraction_config.py",
    log_file=log_file,
    console_output=True,  # Show output to debug what's happening
)

print(f"\n✓ Extraction complete! Logs saved to {log_file}")
print(f"\nExtraction summary:")
for key, value in result.items():
    print(f"  {key}: {value}")



In [ ]:

# Check the logs to see what happened during extraction
import json
from pathlib import Path

log_file = Path(tuni_emotion_extraction.output_dir) / "tuni_extraction_run_log.jsonl"

print(f"Log file: {log_file}")
print(f"File exists: {log_file.exists()}")
print(f"File size: {log_file.stat().st_size if log_file.exists() else 'N/A'} bytes")

if log_file.exists():
    print("\n--- Last 20 log entries ---")
    with open(log_file) as f:
        lines = f.readlines()
        for line in lines[-20:]:
            entry = json.loads(line.strip())
            stage = entry.get("stage", "?")
            level = entry.get("level", "?")
            message = entry.get("message", "")[:80]  # First 80 chars
            print(f"[{level:5s}] {stage:25s} | {message}")


In [ ]:

# First, check what output files already exist
from pathlib import Path
output_dir = Path(tuni_emotion_extraction.output_dir)

print("Existing parquet files:")
for f in sorted(output_dir.glob("*.parquet")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  - {f.name:60s} ({size_mb:.1f} MB)")

print("\nExisting checkpoint files:")
checkpoints = list(output_dir.glob("*.checkpoint.jsonl"))
if checkpoints:
    for cp in sorted(checkpoints):
        print(f"  - {cp.name}")
else:
    print("  (none - these get cleaned up after successful extraction)")

print("\nExisting HDF5 files:")
h5_files = list(output_dir.glob("*.h5"))
if h5_files:
    for h5 in sorted(h5_files):
        print(f"  - {h5.name}")
else:
    print("  (none - these get cleaned up after successful extraction)")


## 3. Explore results


In [ ]:
from scripts.utils import load_parquet  
# Load extraction results

output_dir = Path(tuni_emotion_extraction.output_dir)

# List all output files
output_files = list(output_dir.glob("*.parquet"))
print("Generated output files:")
for f in output_files:
    print(f"  - {f.name}")


In [ ]:
df_opensmile = load_parquet(
    os.path.join(output_dir, "tuni_emotion_dataset_opensmile-compare-2016_3.0s.parquet")
)

print(f"\nOpenSmile embeddings shape: {df_opensmile.shape}")
print(f"Columns: {list(df_opensmile.columns)}")
print(f"\nFirst few rows:")
df_opensmile.head()


In [ ]:
from scripts.feature_visualisation import prepare_and_plot_lda

prepare_and_plot_lda(
    df_opensmile,
    class_col="emotion",
    mode="3d",
    data_type="OpenSmile by Emotion (5-way)",
    filters={
        "emotion": [
            "joy",
            "sadness",
            "anger",
            "gentleness",
            "neutral",
        ]
    },
)


In [ ]:
prepare_and_plot_lda(
    df_opensmile,
    class_col="emotion",
    mode="3d",
    data_type="OpenSmile by Emotion (Classical)",
    filters={"genre": ["classical"]},
)

prepare_and_plot_lda(
    df_opensmile,
    class_col="emotion",
    mode="3d",
    data_type="OpenSmile by Emotion (Pop)",
    filters={"genre": ["pop"]},
)


In [ ]:
df_whisper = load_parquet(
    os.path.join(output_dir, "tuni_emotion_dataset_whisper_whisper-base_3.0s.parquet")
)

prepare_and_plot_lda(
    df_whisper,
    class_col="emotion",
    mode="3d",
    data_type="Whisper by Emotion (5-way)",
    filters={
        "emotion": [
            "joy",
            "sadness",
            "anger",
            "gentleness",
            "neutral",
        ]
    },
)

prepare_and_plot_lda(
    df_whisper,
    class_col="emotion",
    mode="3d",
    data_type="Whisper by Emotion (Classical)",
    filters={"genre": ["classical"]},
)

prepare_and_plot_lda(
    df_whisper,
    class_col="emotion",
    mode="3d",
    data_type="Whisper by Emotion (Pop)",
    filters={"genre": ["pop"]},
)
